In [94]:
# 경로 설정
from course_utils.paths import get_project_root, get_data_dir
import pandas as pd
print("프로젝트 루트 :", get_project_root())
print("데이터 폴더 :", get_data_dir())

RAW_DIR = get_project_root() / "data" / "raw"
PROCESSED_DIR = get_project_root() / "data" / "processed"
REPORT_DIR = get_project_root() / "reports"
PROJECT_ROOT = get_project_root()

프로젝트 루트 : C:\dev\kant-axagent-study\llm-data-analysis-course
데이터 폴더 : C:\dev\kant-axagent-study\llm-data-analysis-course\data


In [95]:
import pandas as pd
import numpy as np

customers = pd.read_csv(get_data_dir()/"raw"/"customers.csv")
orders = pd.read_csv(get_data_dir()/"raw"/"orders.csv")
order_items = pd.read_csv(get_data_dir()/"raw"/"order_items.csv")
products = pd.read_csv(get_data_dir()/"raw"/"products.csv")

print(customers.head())
print(orders.head())
print(order_items.head())
print(products.head())

   customer_id name gender  age city signup_date
0            1  김수민      F   19   광주  2024-06-19
1            2  김정호      F   32   대구  2025-11-02
2            3  이경수      F   61   성남  2024-06-12
3            4  조영호      F   55   울산  2026-04-13
4            5  이예원      F   19   부산  2024-09-13
   order_id  customer_id  order_date payment_method order_status
0         1          123  2026-05-07           card    completed
1         2           77  2025-07-23      naver_pay    cancelled
2         3          138  2025-11-19  bank_transfer    cancelled
3         4           57  2026-01-30      kakao_pay    cancelled
4         5          125  2025-12-21           card    cancelled
   order_item_id  order_id  product_id  quantity  unit_price
0              1         1         100         3      102000
1              2         1          87         5       25000
2              3         1           7         3      142000
3              4         1           9         3      193000
4          

In [96]:
# 원본 구조 Evidence를 만든다. 
raw_data = {
    "customers": customers,
    "order_items": order_items,
    "orders": orders,
    "products": products
}
print(raw_data)

{'customers':      customer_id name gender  age city signup_date
0              1  김수민      F   19   광주  2024-06-19
1              2  김정호      F   32   대구  2025-11-02
2              3  이경수      F   61   성남  2024-06-12
3              4  조영호      F   55   울산  2026-04-13
4              5  이예원      F   19   부산  2024-09-13
..           ...  ...    ...  ...  ...         ...
145          146  김숙자      M   61   성남  2025-12-23
146          147  이정남      M   19   부산  2025-02-11
147          148  오도현      M   29   고양  2026-06-15
148          149  김정자      M   20   부산  2024-10-19
149          150  조미영      M   40   대전  2025-12-04

[150 rows x 6 columns], 'order_items':      order_item_id  order_id  product_id  quantity  unit_price
0                1         1         100         3      102000
1                2         1          87         5       25000
2                3         1           7         3      142000
3                4         1           9         3      193000
4                5 

In [97]:
summary_list = []
for name, frame in raw_data.items():
    # print(key)
    info = {
        "dataset": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "missing_values": int(frame.isna().sum().sum()),
        "missing_duplicated_rows": int(frame.duplicated().sum()),
    }
    summary_list.append(info)
    
summary_list

[{'dataset': 'customers',
  'rows': 150,
  'columns': 6,
  'missing_values': 0,
  'missing_duplicated_rows': 0},
 {'dataset': 'order_items',
  'rows': 764,
  'columns': 5,
  'missing_values': 0,
  'missing_duplicated_rows': 0},
 {'dataset': 'orders',
  'rows': 300,
  'columns': 5,
  'missing_values': 0,
  'missing_duplicated_rows': 0},
 {'dataset': 'products',
  'rows': 100,
  'columns': 4,
  'missing_values': 0,
  'missing_duplicated_rows': 0}]

In [98]:
from src.preprocessing import (

    compare_shapes,

    preprocess_sales_data,

    validate_relationships,

)

processed_data = preprocess_sales_data(raw_data)

preprocessing_comparison = compare_shapes(

    raw_data,

    processed_data,

)

relationship_checks = validate_relationships(

    processed_data

)

In [99]:
from src.preprocessing import compare_shapes, preprocess_sales_data

processed_data = preprocess_sales_data(raw_data)
preprocessing_comparison = compare_shapes(raw_data, processed_data)

print(preprocessing_comparison)

       dataset  rows_raw  columns_raw  rows_processed  columns_processed
0    customers       150            6             150                  6
1  order_items       764            5             764                  6
2       orders       300            5             300                  7
3     products       100            4             100                  4


In [100]:
print(raw_data["order_items"].columns)
print(processed_data["order_items"].columns)

Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price'], dtype='str')
Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price',
       'line_total'],
      dtype='str')


In [101]:
print(raw_data["orders"].columns)
print(processed_data["orders"].columns)

Index(['order_id', 'customer_id', 'order_date', 'payment_method',
       'order_status'],
      dtype='str')
Index(['order_id', 'customer_id', 'order_date', 'payment_method',
       'order_status', 'order_month', 'order_dayofweek'],
      dtype='str')


In [102]:
print(processed_data["orders"].head())

   order_id  customer_id order_date payment_method order_status order_month  \
0         1          123 2026-05-07           card    completed     2026-05   
1         2           77 2025-07-23      naver_pay    cancelled     2025-07   
2         3          138 2025-11-19  bank_transfer    cancelled     2025-11   
3         4           57 2026-01-30      kakao_pay    cancelled     2026-01   
4         5          125 2025-12-21           card    cancelled     2025-12   

  order_dayofweek  
0        Thursday  
1       Wednesday  
2       Wednesday  
3          Friday  
4          Sunday  


In [103]:
print(processed_data["orders"]["order_dayofweek"].value_counts())

order_dayofweek
Saturday     50
Tuesday      47
Monday       47
Wednesday    44
Sunday       43
Friday       35
Thursday     34
Name: count, dtype: int64


키 값 체크

In [104]:
key_map = {
    "customers": "customer_id",
    "products": "product_id",
    "orders":"order_id",
    "order_items":"order_item_id",
}

PK 중복 체크, 결측치 체크

In [105]:
kp_checks = []
for dataset, key in key_map.items():
    frame = processed_data[dataset]
    missing_count = int(frame[key].isna().sum())
    duplicate_count = int(frame[key].duplicated().sum())
    status = ""
    if missing_count == 0 and duplicate_count == 0:
        status = "PASS"
    else:
        status = "FAIL"
    
    kp_checks.append(
        {"dataset": dataset,
        "key": key,
        "missing_count": missing_count,
        "duplicate_count": duplicate_count,
        "status": status
        }
    )



In [106]:
kp_checks

[{'dataset': 'customers',
  'key': 'customer_id',
  'missing_count': 0,
  'duplicate_count': 0,
  'status': 'PASS'},
 {'dataset': 'products',
  'key': 'product_id',
  'missing_count': 0,
  'duplicate_count': 0,
  'status': 'PASS'},
 {'dataset': 'orders',
  'key': 'order_id',
  'missing_count': 0,
  'duplicate_count': 0,
  'status': 'PASS'},
 {'dataset': 'order_items',
  'key': 'order_item_id',
  'missing_count': 0,
  'duplicate_count': 0,
  'status': 'PASS'}]

병합

In [107]:
order_sales = order_items.merge(

    orders[

        [
            "order_id",
            "customer_id",
            "order_date",
            "order_status",
        ]

    ],

    on="order_id",

    how="left",

    validate="many_to_one",

    indicator=True,

)

In [108]:
print("병합 전 행 수: ", len(order_items))
print("병합 후 행 수: ", len(order_sales))
order_sales["_merge"].value_counts(dropna = False)

병합 전 행 수:  764
병합 후 행 수:  764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

In [109]:
order_sales

,order_item_id,order_id,product_id,quantity,unit_price,customer_id,order_date,order_status,_merge
0,1,1,100,3,102000,123,2026-05-07,completed,both
1,2,1,87,5,25000,123,2026-05-07,completed,both
2,3,1,7,3,142000,123,2026-05-07,completed,both
3,4,1,9,3,193000,123,2026-05-07,completed,both
4,5,2,72,4,189000,77,2025-07-23,cancelled,both
...,...,...,...,...,...,...,...,...,...
759,760,297,42,4,28000,22,2026-01-15,completed,both
760,761,298,40,4,174000,64,2025-12-25,cancelled,both
761,762,299,8,2,189000,135,2025-08-08,completed,both
762,763,299,12,4,175000,135,2025-08-08,completed,both


In [110]:
expected_line_total = order_items["quantity"] * order_items["unit_price"]
expected_line_total

0      306000
1      125000
2      426000
3      579000
4      756000
        ...  
759    112000
760    696000
761    378000
762    700000
763    160000
Length: 764, dtype: int64

In [111]:
if "line_total" not in order_items.columns:
    order_items["line_total"] = expected_line_total

order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 764 entries, 0 to 763
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   order_item_id  764 non-null    int64
 1   order_id       764 non-null    int64
 2   product_id     764 non-null    int64
 3   quantity       764 non-null    int64
 4   unit_price     764 non-null    int64
 5   line_total     764 non-null    int64
dtypes: int64(6)
memory usage: 35.9 KB


In [112]:
completed_order_sales = order_sales.loc[
    order_sales["order_status"].eq("completed")
]

print(completed_order_sales.head())
print(completed_order_sales["order_status"].value_counts())

    order_item_id  order_id  product_id  quantity  unit_price  customer_id  \
0               1         1         100         3      102000          123   
1               2         1          87         5       25000          123   
2               3         1           7         3      142000          123   
3               4         1           9         3      193000          123   
12             13         6          83         3       24000           87   

    order_date order_status _merge  
0   2026-05-07    completed   both  
1   2026-05-07    completed   both  
2   2026-05-07    completed   both  
3   2026-05-07    completed   both  
12  2026-03-21    completed   both  
order_status
completed    474
Name: count, dtype: int64


In [120]:
completed_order_sales["line_total"] = (
    completed_order_sales["quantity"]
    * completed_order_sales["unit_price"]
)

In [113]:
print("전체 주문 수:", order_sales["order_item_id"].value_counts().sum())
print("전체 주문 건수(completed):", len(completed_order_sales))
print("전체 주문 건수(completed 외): ", len(order_sales) - len(completed_order_sales))

전체 주문 수: 764
전체 주문 건수(completed): 474
전체 주문 건수(completed 외):  290


In [114]:
order_sales["order_status"].value_counts()

order_status
completed    474
cancelled    162
refunded     128
Name: count, dtype: int64

In [119]:
completed_order_sales.columns

Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price',
       'customer_id', 'order_date', 'order_status', '_merge'],
      dtype='str')

In [ ]:
# product_id 기준으로 merge 하기 -> 완성된 카테고리 sales 
category_sales = completed_order_sales.merge(
    products[],
    on = "product_id",
    how = "left",
    validate = "many_to_one"
)
.groupby("category", as_index = False)
.agg(
    total_quantity = ("quantity","sum")
    total_sales = ("line_total", "sum")
)
.sort_values

     order_item_id  order_id  product_id  quantity  unit_price  customer_id  \
0                1         1         100         3      102000          123   
1                2         1          87         5       25000          123   
2                3         1           7         3      142000          123   
3                4         1           9         3      193000          123   
4               13         6          83         3       24000           87   
..             ...       ...         ...       ...         ...          ...   
469            758       296          40         1      174000          116   
470            759       297          20         5       80000           22   
471            760       297          42         4       28000           22   
472            762       299           8         2      189000          135   
473            763       299          12         4      175000          135   

     order_date order_status _merge  line_total pro

In [124]:
order_status_sales = (
    completed_order_sales
    .groupby("order_status", as_index = False)
    .agg(
        total_quantity = ("quantity", "sum"),
        total_sales = ("line_total", "sum")
    )
    .sort_values("total_sales", ascending = False)
)

print(order_status_sales)

  order_status  total_quantity  total_sales
0    completed            1442    148990000


In [ ]:
print(completed_order_sales.columns)
print(products.head())

,product_id,product_name,category,price
0,1,전자기기 상품 001,전자기기,160000
1,2,도서 상품 002,도서,34000
2,3,전자기기 상품 003,전자기기,152000
3,4,생활용품 상품 004,생활용품,70000
4,5,식품 상품 005,식품,186000
